# Economic Dispatch

The economic dispatch finds the cheapest way to meet demand with the generators and storage that already exist. There is no investment and no binary unit commitment: it is a pure operation problem solved as a linear program.

We start from the 9-node case used in notebook 01 and switch off all investment decisions.

## 1. Set up a working copy of the case

In [1]:
import os, shutil
import pandas as pd

DIR = "work_ED"          # parent folder that will hold the case
CaseName = "9n"      # we reuse the 9-node case from notebook 01

if os.path.exists(DIR):
    shutil.rmtree(DIR)
shutil.copytree(CaseName, os.path.join(DIR, CaseName))

# A coarse time resolution keeps the run fast for this tutorial.
param = os.path.join(DIR, CaseName, "oT_Data_Parameter_9n.csv")
df = pd.read_csv(param)
df.loc[:, "TimeStep"] = 24
df.to_csv(param, index=False)
print("Working copy of the 9n case is ready in", DIR)

Working copy of the 9n case is ready in work_ED


## 2. Switch off investments

In `oT_Data_Option` the investment indicators take the value `2` to *ignore* investments (`0` would keep them as continuous decisions, `1` as binary). With investments ignored and no binary commitment, the model solves a plain economic dispatch.

In [2]:
opt = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"))
opt.loc[0, ["IndBinGenInvest", "IndBinGenRetirement", "IndBinNetInvest"]] = 2
opt.to_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"), index=False)
opt

,IndBinGenInvest,IndBinGenRetirement,IndBinRsrInvest,IndBinNetInvest,IndBinNetH2Invest,IndBinNetHeatInvest,IndBinGenOperat,IndBinNetLosses,IndBinLineCommit,IndBinSingleNode,IndBinGenRamps,IndBinGenMinTime
0,2,2,0,2,0,0,0,1,0,0,0,0


## 3. Run the model

In [3]:
from openTEPES.openTEPES import openTEPES_run

model = openTEPES_run(DIR, CaseName, "appsi_highs", "Yes", "No")
print("Total system cost [MEUR]:", round(model.vTotalSCost(), 3))

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****


Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****
Ramp and min up/down time  constraints ****


Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
  Total system                 cost [MEUR]  195.82048270540588  Constraints 41136  Variables 50965  Seconds 2
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  0.0
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  195.81881871299143
  Total consumption operation  cost [MEUR]  6.521204713326766e-05
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.0015987803675333787
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s


Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s
Writing elect network summary results  ...  0 s
Writing           reliability indexes  ...  0 s
Writing           flexibility results  ...  0 s


/private/tmp/claude-501/-Users-philias-ai-research-repos-openTEPES-tutorial/8f6d4ac5-9ff1-45bc-8a3c-5cc562abe3f1/scratchpad/venv312/lib/python3.12/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


Writing  generation operation results  ...  0 s
Writing         ESS operation results  ...  0 s
Writing elect netwk operation results  ...  0 s


Writing  marginal information results  ...  0 s


Writing              economic results  ...  0 s
Plotting electricity network     maps  ...  0 s
Total system cost [MEUR]: 195.82


## 4. Read a result

Generation per technology and load level is written to the case folder.

In [4]:
tech = pd.read_csv(os.path.join(DIR, CaseName, "oT_Result_TechnologyGeneration_9n.csv"))
tech.head()

,Period,Scenario,LoadLevel,Coal,ESS,Gas,Nuclear,Oil,RES
0,2030,sc01,01-01 23:00:00+01:00,0.0,0.000000,222.014114,677.697386,0.0,110.487024
1,2030,sc01,01-02 23:00:00+01:00,0.0,0.000000,465.949315,677.906264,0.0,75.353387
2,2030,sc01,01-03 23:00:00+01:00,0.0,8.703474,497.855291,669.269005,0.0,108.613514
3,2030,sc01,01-04 23:00:00+01:00,0.0,15.824705,490.827537,662.155335,0.0,122.583894
4,2030,sc01,01-05 23:00:00+01:00,0.0,0.000000,547.230234,677.963887,0.0,53.285052
